In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/zalo-dataset/corpus.csv
/kaggle/input/zalo-dataset/crawled_test_5k.jsonl


In [2]:
!pip install protobuf==3.20.3
!pip install sentence-transformers==4.1.0 
!pip install faiss-gpu-cu12==1.12.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-me

In [3]:
df = pd.read_csv('/kaggle/input/zalo-dataset/corpus.csv')

/tmp/ipykernel_20/1028339124.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/kaggle/input/zalo-dataset/corpus.csv')


In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import torch
import os # Import os for path manipulation
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device for embedding: {device}")
embedding_model = SentenceTransformer('bkai-foundation-models/vietnamese-bi-encoder', device=device)

if device == 'cuda':
     embedding_model.half() # Convert model to half precision for faster GPU inference

# Filter out any empty or invalid chunks from the DataFrame column
valid_chunks = (
    "Title: " + df['title'].astype(str) + 
    ". Content: " + df['text'].astype(str)
).tolist()

with torch.no_grad():
     doc_embeddings = embedding_model.encode(
         valid_chunks,
         batch_size=512,
         convert_to_numpy=True
     )

doc_embeddings = doc_embeddings.astype('float32')

dimension = doc_embeddings.shape[1]

doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

if device == 'cuda' and len(valid_chunks) > 0:
     print("Creating FAISS index on GPU.")
     res = faiss.StandardGpuResources() # Use a single GPU
     cpu_index = faiss.IndexFlatIP(dimension) # Create CPU index first
     gpu_index = faiss.index_cpu_to_gpu(res, 0, cpu_index) # Move to GPU (ID 0)
     gpu_index.add(doc_embeddings)
     # Convert back to CPU index before saving
     final_index = faiss.index_gpu_to_cpu(gpu_index)

else:
     print("Creating FAISS index on CPU.")
     final_index = faiss.IndexFlatIP(dimension)
     final_index.add(doc_embeddings)

# --- Saving the Index ---
# Define a filename for your index
index_filename = "index.faiss"

# Define the directory where you want to save it (e.g., /kaggle/working/ for Kaggle output)
output_dir = "/kaggle/working/" # This is the standard output directory in Kaggle
save_path = os.path.join(output_dir, index_filename)

print(f"Saving FAISS index to: {save_path}")
faiss.write_index(final_index, save_path)
print("FAISS index saved successfully!")

2026-01-01 14:28:59.562954: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767277739.928370      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767277740.036009      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device for embedding: cuda


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

Creating FAISS index on GPU.
Saving FAISS index to: /kaggle/working/index.faiss
FAISS index saved successfully!
